# Boucle complète : données synthétiques, validation et sauvegarde

Réunir les briques déjà expliquées dans une expérience indépendante et reproductible.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#boucle-entrainement-complete).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


La petite boucle savait ajuster une règle sur nos trois exemples fixes. Voici une autre expérience, plus longue : classer deux nuages de points, charger les données par lots et vérifier un rechargement.

Les trois colonnes s’appellent maintenant x1, x2 et x3. Ce sont des coordonnées synthétiques, pas RMS, facteur de crête ou descripteur acoustique. Les classes sont nuage 0 et nuage 1.

| Objet | Choix | Sens |
| --- | --- | --- |
| Classe 0 | Centre (−1,−1,−1) | Coordonnées autour du premier centre |
| Classe 1 | Centre (1,1,1) | Coordonnées autour du second centre |
| Dispersion | Bruit gaussien ×0,4 | Écart-type 0,4 sur chaque coordonnée |
| Train / validation | 128 / 64 exemples | Deux tirages distincts |
| Lots | 32 exemples au maximum | Un lot fixe de coordonnées par passage |

La graine fixe une suite de tirages sur cet environnement. Elle ne transforme pas ces points en données audio ni en résultat général sur la classification musicale.

### Comprendre d’abord la fabrication d’un point

Une cible 0 devient un signe −1 ; une cible 1 devient +1. unsqueeze(1) transforme ces signes en une colonne, ensuite diffusée sur trois coordonnées.

Cette miniature choisit son bruit à la main : le premier point reçoit [0 ; 0,5 ; −0,5]×0,4. Il devient [−1 ; −0,8 ; −1,2] autour du centre négatif.

## Démonstration 1 · Séparer centre, écart et mesure

La version complète remplacera seulement le bruit choisi par des tirages gaussiens.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

Les trois classes et les trois lignes de bruit sont fixées à la main. Rien n’est encore tiré aléatoirement dans cette miniature.

In [4]:
cibles = torch.tensor([0, 1, 0], dtype=torch.long)
bruit_choisi = torch.tensor([[0., 0.5, -0.5],
                             [0.5, 0., -0.5],
                             [-0.5, 0., 0.5]])


In [5]:
observer('cibles', cibles)
observer('bruit_choisi', bruit_choisi)

cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]
bruit_choisi forme (3, 3) type torch.float32 requires_grad False
  valeurs : [[0.0, 0.5, -0.5], [0.5, 0.0, -0.5], [-0.5, 0.0, 0.5]]


### Calculer l’état suivant

float convertit les indices pour le calcul. La formule 2×cible−1 associe 0 à −1 et 1 à +1.

In [6]:
signes = 2 * cibles.float() - 1


In [7]:
observer('signes', signes)

signes forme (3,) type torch.float32 requires_grad False
  valeurs : [-1.0, 1.0, -1.0]


### Calculer l’état suivant

unsqueeze(1) transforme les trois signes en une colonne (3,1). Chaque signe servira aux trois coordonnées du même point.

In [8]:
centres = signes.unsqueeze(1)


In [9]:
observer('centres', centres)

centres forme (3, 1) type torch.float32 requires_grad False
  valeurs : [[-1.0], [1.0], [-1.0]]


### Calculer l’état suivant

Le bruit choisi est multiplié par 0,4. L’addition diffuse ensuite le centre sur les trois colonnes de la ligne.

In [10]:
ecarts = 0.4 * bruit_choisi
mesures = centres + ecarts


In [11]:
observer('ecarts', ecarts)
observer('mesures', mesures)

ecarts forme (3, 3) type torch.float32 requires_grad False
  valeurs : [[0.0, 0.20000000298023224, -0.20000000298023224], [0.20000000298023224, 0.0, -0.20000000298023224], [-0.20000000298023224, 0.0, 0.20000000298023224]]
mesures forme (3, 3) type torch.float32 requires_grad False
  valeurs : [[-1.0, -0.800000011920929, -1.2000000476837158], [1.2000000476837158, 1.0, 0.800000011920929], [-1.2000000476837158, -1.0, -0.800000011920929]]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [12]:
print("centres :", centres.tolist())
print("ecarts :", ecarts)
print("mesures :", mesures)
print("formes :", tuple(centres.shape), tuple(mesures.shape))

centres : [[-1.0], [1.0], [-1.0]]
ecarts : tensor([[ 0.0000,  0.2000, -0.2000],
        [ 0.2000,  0.0000, -0.2000],
        [-0.2000,  0.0000,  0.2000]])
mesures : tensor([[-1.0000, -0.8000, -1.2000],
        [ 1.2000,  1.0000,  0.8000],
        [-1.2000, -1.0000, -0.8000]])
formes : (3, 1) (3, 3)


Changer de classe déplace le centre ; changer le bruit déplace le point autour de ce centre.

### Le programme complet, après cette miniature

Dans fabriquer(n), randn(n,3) produit maintenant un bruit gaussien. Le programme recommence sa graine, puis fabrique séparément les 128 points d’entraînement et les 64 de validation.

L’assemblage personnalisé serait facultatif pour ces formes fixes : le comportement par défaut pourrait empiler les exemples. Nous le gardons explicite pour rendre son contrat inspectable.

Le modèle Sequential applique Linear(3,8), ReLU puis Linear(8,2). La couche cachée produit huit caractéristiques ; la dernière les transforme en deux scores de classes.

## Démonstration 2 · Relier les morceaux sans sauter leur ordre

Ce programme crée modele-nuages.pt dans le dossier courant.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [13]:
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader



### Calculer l’état suivant

Nous choisissons CPU, une graine et un thread. La perte de deux scores égaux donne un contrôle connu : −ln(0,5)≈0,693147.

In [14]:
torch.manual_seed(17)
torch.set_num_threads(1)
device = torch.device("cpu")
perte_neutre = nn.functional.cross_entropy(torch.zeros(1, 2), torch.tensor([0]))


In [15]:
observer('device', device)
observer('perte_neutre', perte_neutre)

perte_neutre forme () type torch.float32 requires_grad False
  valeurs : 0.6931471824645996


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [16]:
print(f"Perte de deux scores égaux : {perte_neutre.item():.6f}")



Perte de deux scores égaux : 0.693147


### Définir la règle avant de l’utiliser

arange(n)%2 alterne les classes. Le calcul des centres reprend la miniature, et randn remplace le bruit fixé à la main.

In [17]:
def fabriquer(n):
    cibles = torch.arange(n) % 2
    centres = (2 * cibles.float() - 1).unsqueeze(1)
    mesures = 0.4 * torch.randn(n, 3) + centres
    return mesures, cibles



### Définir la règle avant de l’utiliser

Nuages garde mesures et cibles. __len__ donne l’effectif ; __getitem__ utilise le même indice pour les deux objets.

In [18]:
class Nuages(Dataset):
    def __init__(self, mesures, cibles):
        self.mesures, self.cibles = mesures, cibles
    def __len__(self):
        return len(self.cibles)
    def __getitem__(self, indice):
        return self.mesures[indice], self.cibles[indice]



### Définir la règle avant de l’utiliser

zip(*) rassemble les mesures d’un côté et les cibles de l’autre. stack crée l’axe des exemples ; long explicite le type entier des classes.

In [19]:
def assembler(exemples):
    mesures, cibles = zip(*exemples)
    return torch.stack(mesures), torch.stack(cibles).long()



### Calculer l’état suivant

fabriquer renvoie un couple. L’étoile de Nuages(*fabriquer(...)) fournit ces deux objets aux deux arguments de son constructeur.

Cette écriture équivaut à mesures,cibles=fabriquer(128), puis train=Nuages(mesures,cibles). Elle n’ajoute aucun exemple.

In [20]:
train = Nuages(*fabriquer(128))
validation = Nuages(*fabriquer(64))
train_loader = DataLoader(train, batch_size=32, shuffle=True,
                          collate_fn=assembler, num_workers=0)
val_loader = DataLoader(validation, batch_size=32, shuffle=False,
                        collate_fn=assembler, num_workers=0)



In [21]:
observer('train', train)
observer('validation', validation)
observer('train_loader', train_loader)
observer('val_loader', val_loader)

### Définir la règle avant de l’utiliser

Sequential compose les couches. CrossEntropyLoss attend les deux scores et la classe entière. Adam reçoit tous les paramètres du modèle.

In [22]:
def creer_modele():
    return nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 2))



### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [23]:
modele = creer_modele().to(device)
critere = nn.CrossEntropyLoss()
optimiseur = torch.optim.Adam(modele.parameters(), lr=0.03)
historique = []


In [24]:
observer('historique', historique)

historique = []


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [25]:
N = 5


In [26]:
observer('N', N)

N = 5


### Répéter la même étape sur les données

train choisit le mode du modèle. somme et effectif repartent à zéro pour cette époque ; le chargeur fournit les lots successifs.

Les données rejoignent le device choisi. Nous remettons les gradients à zéro, calculons scores puis perte, dérivons, et seulement ensuite appelons step.

Chaque perte est une moyenne de lot. Multiplier par len(cibles) retrouve sa somme ; effectif compte les exemples pour la division finale.

eval choisit le mode. no_grad encadre les nouveaux calculs ; cette boucle ne contient ni backward ni step. Ses compteurs sont réinitialisés.

In [27]:
for epoch in range(1, N + 1):
    modele.train()
    somme, effectif = 0.0, 0
    for mesures, cibles in train_loader:
        mesures, cibles = mesures.to(device), cibles.to(device)
        optimiseur.zero_grad(set_to_none=True)
        scores = modele(mesures)
        perte = critere(scores, cibles)
        perte.backward()
        optimiseur.step()
        somme += perte.item() * len(cibles)
        effectif += len(cibles)
    train_loss = somme / effectif

    modele.eval()
    somme, effectif = 0.0, 0
    with torch.no_grad():
        for mesures, cibles in val_loader:
            mesures, cibles = mesures.to(device), cibles.to(device)
            scores = modele(mesures)
            perte = critere(scores, cibles)
            somme += perte.item() * len(cibles)
            effectif += len(cibles)
    val_loss = somme / effectif
    historique.append([train_loss, val_loss])
    print(f"epoch {epoch}: train={train_loss:.6f}, val={val_loss:.6f}")



epoch 1: train=0.783676, val=0.444391
epoch 2: train=0.325945, val=0.156877
epoch 3: train=0.114880, val=0.050986
epoch 4: train=0.040490, val=0.017714
epoch 5: train=0.014973, val=0.007170


### Calculer l’état suivant

Le dictionnaire model contient les poids. La liste classes rappelle l’ordre des deux sorties, indispensable pour interpréter leurs indices.

In [28]:
chemin = Path("modele-nuages.pt")
torch.save({"model": modele.state_dict(), "classes": ["nuage 0", "nuage 1"]}, chemin)
recharge = creer_modele().to(device)
etat = torch.load(chemin, map_location=device, weights_only=True)


In [29]:
observer('chemin', chemin)
observer('recharge', recharge)
observer('etat', etat)

etat : clés ['model', 'classes']


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [30]:
recharge.load_state_dict(etat["model"])
recharge.eval()


Sequential(
  (0): Linear(in_features=3, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=2, bias=True)
)

### Appliquer le contexte explicite

La sonde est constituée des deux premières lignes de validation. torch.equal compare les scores des modèles avant et après chargement.

In [31]:
with torch.no_grad():
    sonde = validation.mesures[:2].to(device)
    identique = torch.equal(modele(sonde), recharge(sonde))


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [32]:
print("Rechargement identique :", identique)

Rechargement identique : True


Nous avons relié un entraînement, une observation séparée et un rechargement vérifié. Les pertes restent celles de cette expérience synthétique.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Assemblez les mesures des deux paires en une matrice : une ligne par extrait, trois coordonnées par ligne.

In [33]:
exo_exemples=[(torch.tensor([1.,2.,3.]),0),(torch.tensor([4.,5.,6.]),1)]
exo_entrees = ...  # Remplacer les points par votre réponse.
if exo_entrees is Ellipsis or exo_entrees is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_entrees.shape) == (2,3)
    assert exo_entrees.tolist() == [[1,2,3],[4,5,6]]
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Chaque paire contient un vecteur puis sa cible. stack ajoute l’axe des exemples.

</details>

### Exercice 2

Construisez les indices de classe dans le même ordre, au format demandé par CrossEntropyLoss.

In [34]:
exo_exemples=[(torch.tensor([1.,2.,3.]),0),(torch.tensor([4.,5.,6.]),1)]
exo_cibles = ...  # Remplacer les points par votre réponse.
if exo_cibles is Ellipsis or exo_cibles is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_cibles.dtype == torch.int64
    assert exo_cibles.tolist() == [0,1]
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Le second élément de chaque paire est l’annotation. Ne triez pas les cibles séparément des mesures.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_exemples=[(torch.tensor([1.,2.,3.]),0),(torch.tensor([4.,5.,6.]),1)]
exo_entrees = torch.stack([exemple[0] for exemple in exo_exemples])
if exo_entrees is Ellipsis or exo_entrees is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_entrees.shape) == (2,3)
    assert exo_entrees.tolist() == [[1,2,3],[4,5,6]]
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_exemples=[(torch.tensor([1.,2.,3.]),0),(torch.tensor([4.,5.,6.]),1)]
exo_cibles = torch.tensor([exemple[1] for exemple in exo_exemples],dtype=torch.long)
if exo_cibles is Ellipsis or exo_cibles is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_cibles.dtype == torch.int64
    assert exo_cibles.tolist() == [0,1]
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [35]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()